[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommy-Burns/easy-eo/blob/main/examples/00_getting_started/01_installation_and_setup.ipynb)

# Installation and setup

Check that Easy-EO and its geospatial stack are installed correctly, see which optional extras are available, and warm up the sample data every other notebook in this series uses.

- **Data:** the bundled sample (downloaded on first use, then cached)
- **Network:** needed once, to download the sample; nothing else

In [ ]:
# Running in Colab? Install Easy-EO. Does nothing anywhere else.
import sys

if "google.colab" in sys.modules:
    %pip install -q "easy-eo[stac,xarray]"


## Install

Easy-EO is on PyPI and on conda-forge. The core install pulls in Rasterio, GeoPandas, NumPy and Matplotlib:

```bash
pip install easy-eo
# or
conda install -c conda-forge easy-eo
```

Two optional extras unlock features used later in this series:

```bash
pip install "easy-eo[stac]"     # search and load scenes from a STAC catalog
pip install "easy-eo[xarray]"   # to_xarray() / from_xarray() interop
pip install "easy-eo[stac,xarray]"   # both
```

conda has no equivalent of extras - `conda install "easy-eo[stac]"` is not valid syntax - so install the same packages by name, and do not mix package managers within one environment:

```bash
conda install -c conda-forge easy-eo pystac-client planetary-computer   # stac
conda install -c conda-forge easy-eo xarray rioxarray                   # xarray
```

Uncomment the line below to install from inside the notebook.

In [1]:
# %pip install "easy-eo[stac,xarray]"

## Verify the install

`show_versions()` prints Easy-EO's version alongside the versions of the libraries it builds on - GDAL included, which is the one that usually causes trouble. Include this output in any bug report.

In [2]:
import eeo

eeo.show_versions()

Easy-EO version information
easy-eo            : 0.1.0b1
python             : 3.10.19
OS                 : Linux 7.0.0-28-generic
rasterio           : 1.4.3
GDAL               : 3.6.2
numpy              : 1.26.4
geopandas          : 1.1.1
matplotlib         : 3.10.6
xarray             : 2025.4.0
rioxarray          : 0.19.0
dask               : not installed
pystac-client      : 0.9.0
planetary-computer : 1.0.0


## Which optional extras do you have?

Nothing here raises if an extra is missing - Easy-EO only imports an extra's dependencies when you actually call the feature, so `import eeo` stays fast and dependency-free.

In [3]:
from importlib.util import find_spec

extras = {
    "stac": ["pystac_client", "planetary_computer"],
    "xarray": ["xarray", "rioxarray"],
}

for extra, modules in extras.items():
    missing = [m for m in modules if find_spec(m) is None]
    status = "installed" if not missing else f"missing: {', '.join(missing)}"
    print(f"easy-eo[{extra}]  ->  {status}")

easy-eo[stac]  ->  installed
easy-eo[xarray]  ->  installed


## The sample data

Every offline notebook in this series runs on one curated sample: a 1024x1024 Sentinel-2 L2A subset (blue, green, red, nir at 10 m), a Copernicus GLO-30 DEM on the same grid, and a region-of-interest polygon.

`load_sample_dataset()` returns a namespace of the individual files. It is instant and touches no network - each attribute is a *lazy* handle that downloads and checksum-verifies its file only when something actually opens it.

In [4]:
from eeo.datasets import load_sample_dataset

sd = load_sample_dataset()
sd

SampleDataset(sentinel2_stacked, sentinel2_cog_stacked, sentinel2_blue, sentinel2_blue_cog, sentinel2_green, sentinel2_green_cog, sentinel2_red, sentinel2_red_cog, sentinel2_nir, sentinel2_nir_cog, copernicus_dem, copernicus_dem_cog, boundary)

In [5]:
for handle in sd:
    print(f"{handle.name:24s} {handle.kind:7s} {handle.description}")

sentinel2_stacked        raster  Sentinel-2 L2A 4-band stack (blue/green/red/nir), 1024x1024 @ 10 m, EPSG:32633.
sentinel2_cog_stacked    raster  Cloud-Optimized GeoTIFF variant of the 4-band Sentinel-2 stack (HTTP range-read demos).
sentinel2_blue           raster  Sentinel-2 L2A band B02 (blue), 1024x1024 @ 10 m, EPSG:32633.
sentinel2_blue_cog       raster  Cloud-Optimized GeoTIFF variant of the blue band (B02).
sentinel2_green          raster  Sentinel-2 L2A band B03 (green), 1024x1024 @ 10 m, EPSG:32633.
sentinel2_green_cog      raster  Cloud-Optimized GeoTIFF variant of the green band (B03).
sentinel2_red            raster  Sentinel-2 L2A band B04 (red), 1024x1024 @ 10 m, EPSG:32633.
sentinel2_red_cog        raster  Cloud-Optimized GeoTIFF variant of the red band (B04).
sentinel2_nir            raster  Sentinel-2 L2A band B08 (nir), 1024x1024 @ 10 m, EPSG:32633.
sentinel2_nir_cog        raster  Cloud-Optimized GeoTIFF variant of the nir band (B08).
copernicus_dem           raster 

## Download everything up front

Pass `prefetch=True` to download and verify all of the sample files in one go - useful before working offline. It is a few tens of megabytes, and re-running it costs nothing once the files are cached.

In [6]:
sd = load_sample_dataset(prefetch=True)
print("all sample files cached")

all sample files cached


Files land in `~/.cache/easy-eo` by default. Override the location with the `EEO_DATA_DIR` environment variable (`XDG_CACHE_HOME` is honoured too).

In [7]:
from eeo.datasets import cache_dir

path = cache_dir()
print(path)
for f in sorted(path.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size / 1e6:6.1f} MB")

/home/tommy/.cache/easy-eo
  B02.tif                         1.5 MB
  B03.tif                         1.5 MB
  B04.tif                         1.6 MB
  B08.tif                         1.5 MB
  DEM.tif                         3.4 MB
  DEM_COG.tif                     5.4 MB
  roi.gpkg                        0.1 MB
  sentinel2_small.tif             5.8 MB
  sentinel2_small_cog.tif         7.9 MB


## Open one and confirm it works

In [8]:
from eeo import load_raster

scene = load_raster(sd.sentinel2_stacked)
scene.describe()

EEORasterDataset
  source      : /home/tommy/.cache/easy-eo/sentinel2_small.tif
  driver      : GTiff
  bands       : 4
  band names  : 1: blue, 2: green, 3: red, 4: nir
  size        : 1024 × 1024  (height × width)
  dtype       : uint16
  crs         : EPSG:32633 — WGS 84 / UTM zone 33N
  pixel size  : 10 × 10  (CRS units)
  extent      : 377720, 5340230, 387960, 5350470  (minx, miny, maxx, maxy)
  nodata      : 0.0
  timestamp   : none
  attrs       : none


## Attribution

The sample is real satellite data and carries licence obligations. Every handle knows its own provenance - reproduce it if you republish anything derived from the sample.

In [9]:
print(sd.sentinel2_stacked.info())
print()
print(sd.copernicus_dem.info())

sentinel2_stacked (raster)
  Sentinel-2 L2A 4-band stack (blue/green/red/nir), 1024x1024 @ 10 m, EPSG:32633.
  file: sentinel2_small.tif
  attribution: Contains modified Copernicus Sentinel-2 L2A data 2023 (tile T33UUP, acquired 2023-09-07), processed by ESA; accessed via Microsoft Planetary Computer. Licensed under the Copernicus open data terms.

copernicus_dem (raster)
  Copernicus GLO-30 DEM warped onto the Sentinel-2 grid, float32 metres.
  file: DEM.tif
  attribution: Contains modified Copernicus DEM GLO-30 data (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA); accessed via Microsoft Planetary Computer.


In [10]:
scene.close()

## Next

- **[02_quickstart_ndvi](02_quickstart_ndvi.ipynb)** - load, compute NDVI, plot, in about eight lines.